# Aria Monetization & Subscriptions Guide

**Complete guide for implementing subscription tiers, feature gating, billing, and revenue optimization.**

Learn how to build sustainable revenue while keeping Aria accessible to all users.


## Subscription Model

### Tiered Pricing Strategy

```
FREE Tier
├─ Features: Basic chat, 1 model access
├─ Limits: 100 messages/month, 1 conversation
├─ Price: $0
└─ Goal: User acquisition

PRO Tier
├─ Features: All chat models, priority support, custom instructions
├─ Limits: Unlimited messages, 50 conversations, 10GB storage
├─ Price: $9.99/month or $99/year (2 month discount)
└─ Goal: Power users, indie developers

ENTERPRISE Tier
├─ Features: White-label, API access, SLA, dedicated support
├─ Limits: Unlimited everything
├─ Price: Custom (contact sales)
└─ Goal: Teams, companies
```

### Feature Matrix

```python
# config/subscription_tiers.yaml
tiers:
  free:
    name: "Free"
    monthly_price: 0
    annual_price: 0
    features:
      chat_messages_per_month: 100
      conversations: 1
      models: ["openai-gpt3.5"]
      custom_instructions: false
      priority_support: false
      api_access: false
      white_label: false
      storage_gb: 1
      download_conversations: false

  pro:
    name: "Pro"
    monthly_price: 9.99
    annual_price: 99.99
    features:
      chat_messages_per_month: null  # unlimited
      conversations: 50
      models: ["openai-gpt3.5", "openai-gpt4", "claude", "local"]
      custom_instructions: true
      priority_support: true
      api_access: true
      white_label: false
      storage_gb: 10
      download_conversations: true

  enterprise:
    name: "Enterprise"
    monthly_price: null  # custom
    annual_price: null
    features:
      chat_messages_per_month: null
      conversations: null
      models: null  # all
      custom_instructions: true
      priority_support: true
      api_access: true
      white_label: true
      storage_gb: null  # unlimited
      download_conversations: true
      custom_domain: true
      analytics: true
```

---

## Feature Gating Implementation

### Subscription Manager

```python
# shared/subscription_manager.py
from enum import Enum
from typing import Optional

class SubscriptionTier(Enum):
    FREE = "free"
    PRO = "pro"
    ENTERPRISE = "enterprise"

class SubscriptionManager:
    def __init__(self, db_engine):
        self.db = db_engine
        self.tier_config = self._load_config()

    def get_user_tier(self, user_id: str) -> SubscriptionTier:
        """Get user's current subscription tier."""
        query = """
        SELECT tier FROM user_subscriptions
        WHERE user_id = %s AND status = 'active'
        ORDER BY created_at DESC LIMIT 1
        """
        result = self.db.execute(query, (user_id,))
        tier = result.scalar() or "free"
        return SubscriptionTier(tier)

    def check_feature_access(self, user_id: str, feature: str) -> bool:
        """Check if user has access to feature."""
        tier = self.get_user_tier(user_id)
        allowed_tiers = self.tier_config.get(feature, {}).get("allowed_tiers", [])
        return tier.value in allowed_tiers

    def check_limit(self, user_id: str, limit_name: str) -> dict:
        """Check if user has hit limit."""
        tier = self.get_user_tier(user_id)
        limit = self.tier_config[tier.value].get(limit_name)

        if limit is None:  # Unlimited
            return {"remaining": float('inf'), "limit": None, "exceeded": False}

        # Get current usage
        query = f"""
        SELECT COUNT(*) FROM user_usage
        WHERE user_id = %s
        AND DATE_TRUNC('month', created_at) = DATE_TRUNC('month', NOW())
        AND usage_type = %s
        """
        result = self.db.execute(query, (user_id, limit_name))
        current_usage = result.scalar() or 0

        return {
            "limit": limit,
            "current": current_usage,
            "remaining": max(0, limit - current_usage),
            "exceeded": current_usage >= limit
        }

    def upgrade_subscription(self, user_id: str, new_tier: str, billing_period: str = "monthly"):
        """Upgrade user subscription."""
        # Create new subscription
        subscription = {
            "user_id": user_id,
            "tier": new_tier,
            "status": "active",
            "billing_period": billing_period,
            "created_at": datetime.now()
        }

        # Handle existing subscription
        old_tier = self.get_user_tier(user_id)
        if old_tier != SubscriptionTier.FREE:
            # Prorate remaining balance
            prorated = self._calculate_proration(user_id, old_tier, new_tier)
            subscription["proration_credits"] = prorated

        # Save to database
        self.db.insert("user_subscriptions", subscription)

        # Send confirmation email
        self._send_email(user_id, "upgrade_confirmation", {"tier": new_tier})

        return subscription

# Feature gating in FastAPI
@app.route("/api/chat/completions", methods=["POST"])
async def chat_completions(req: func.HttpRequest):
    """Chat endpoint with feature gating."""
    user_id = req.headers.get("X-User-ID")
    subscription = SubscriptionManager(db)

    # Check feature access
    if not subscription.check_feature_access(user_id, "chat_api"):
        return func.HttpResponse(
            json.dumps({
                "error": "Feature not available in your plan",
                "upgrade_url": "https://aria.example.com/upgrade"
            }),
            status_code=403
        )

    # Check rate limit
    limit_check = subscription.check_limit(user_id, "messages_per_month")
    if limit_check["exceeded"]:
        return func.HttpResponse(
            json.dumps({
                "error": f"Monthly message limit reached ({limit_check['limit']})",
                "upgrade_url": "https://aria.example.com/upgrade"
            }),
            status_code=429
        )

    # Process chat
    response = await get_chat_response(req.get_json())
    return func.HttpResponse(json.dumps(response))
```

---

## Billing & Payments

### Stripe Integration

```python
# ai-projects/billing/stripe_integration.py
import stripe
from datetime import datetime, timedelta

stripe.api_key = os.getenv("STRIPE_SECRET_KEY")

class StripeManager:
    def __init__(self):
        self.products = self._init_products()

    def _init_products(self):
        """Initialize Stripe products for each tier."""
        return {
            "pro": {
                "monthly": os.getenv("STRIPE_PRODUCT_PRO_MONTHLY"),
                "annual": os.getenv("STRIPE_PRODUCT_PRO_ANNUAL")
            },
            "enterprise": {
                "custom": "custom_quote"
            }
        }

    def create_checkout_session(self, user_id: str, tier: str, period: str = "monthly"):
        """Create Stripe checkout session."""
        product_id = self.products[tier][period]

        session = stripe.checkout.Session.create(
            customer_email=self._get_user_email(user_id),
            client_reference_id=user_id,
            line_items=[{
                "price": product_id,
                "quantity": 1
            }],
            mode="subscription",
            success_url="https://aria.example.com/billing/success?session={CHECKOUT_SESSION_ID}",
            cancel_url="https://aria.example.com/billing/upgrade",
            metadata={
                "user_id": user_id,
                "tier": tier
            }
        )

        return session.url

    def handle_webhook(self, event: dict):
        """Handle Stripe webhooks."""
        if event["type"] == "customer.subscription.created":
            subscription = event["data"]["object"]
            user_id = subscription["metadata"]["user_id"]
            tier = subscription["metadata"]["tier"]

            # Update database
            SubscriptionManager(db).upgrade_subscription(user_id, tier)

        elif event["type"] == "customer.subscription.deleted":
            subscription = event["data"]["object"]
            user_id = subscription["metadata"]["user_id"]

            # Downgrade to free
            SubscriptionManager(db).downgrade_to_free(user_id)

        elif event["type"] == "invoice.payment_failed":
            # Send retry email
            pass

# Usage in function_app.py
@app.route("/api/billing/subscribe", methods=["POST"])
async def subscribe(req: func.HttpRequest):
    """Start subscription."""
    user_id = req.headers.get("X-User-ID")
    data = req.get_json()
    tier = data.get("tier", "pro")

    stripe_mgr = StripeManager()
    checkout_url = stripe_mgr.create_checkout_session(user_id, tier)

    return func.HttpResponse(
        json.dumps({"checkout_url": checkout_url}),
        status_code=200
    )
```

---

## Monetization Analytics

### Revenue Tracking

```python
# scripts/revenue_analytics.py
import pandas as pd

class RevenueAnalytics:
    def __init__(self, db_url: str):
        self.engine = create_engine(db_url)

    def get_mrr(self) -> float:
        """Calculate Monthly Recurring Revenue."""
        query = """
        SELECT SUM(monthly_value) as mrr
        FROM (
            SELECT
                user_id,
                CASE
                    WHEN tier = 'pro' THEN 9.99
                    WHEN tier = 'enterprise' THEN 99.99
                    ELSE 0
                END as monthly_value
            FROM user_subscriptions
            WHERE status = 'active'
        ) t
        """
        result = pd.read_sql(query, self.engine)
        return result['mrr'].iloc[0] or 0

    def get_churn_rate(self, days: int = 30) -> float:
        """Calculate customer churn rate."""
        query = f"""
        SELECT
            COUNT(DISTINCT CASE
                WHEN status = 'cancelled' THEN user_id
            END)::float / COUNT(DISTINCT user_id) as churn_rate
        FROM user_subscriptions
        WHERE created_at > NOW() - INTERVAL '{days} days'
        """
        result = pd.read_sql(query, self.engine)
        return result['churn_rate'].iloc[0] or 0

    def get_arpu(self) -> float:
        """Calculate Average Revenue Per User."""
        query = """
        SELECT
            SUM(CASE WHEN tier = 'free' THEN 0 WHEN tier = 'pro' THEN 9.99 ELSE 99.99 END)
            / COUNT(DISTINCT user_id) as arpu
        FROM user_subscriptions
        WHERE status = 'active'
        """
        result = pd.read_sql(query, self.engine)
        return result['arpu'].iloc[0] or 0

    def get_ltv(self) -> float:
        """Calculate Lifetime Value."""
        arpu = self.get_arpu()
        churn = self.get_churn_rate()

        if churn == 0:
            return arpu * 120  # Assume 10 year lifetime

        retention = 1 - churn
        lifetime_months = 1 / (1 - retention) if retention < 1 else 120
        return arpu * lifetime_months

    def export_dashboard(self):
        """Export key metrics for dashboard."""
        return {
            "mrr": self.get_mrr(),
            "churn_rate": self.get_churn_rate(),
            "arpu": self.get_arpu(),
            "ltv": self.get_ltv(),
            "tier_distribution": self._get_tier_distribution(),
            "upgrade_funnel": self._get_upgrade_funnel()
        }

    def _get_tier_distribution(self) -> dict:
        """Tier distribution."""
        query = """
        SELECT tier, COUNT(*) as count
        FROM user_subscriptions
        WHERE status = 'active'
        GROUP BY tier
        """
        df = pd.read_sql(query, self.engine)
        return df.to_dict('records')
```

---

## Handling Upgrades & Downgrades

### Upgrade Flow

```python
# Upgrade with proration
def upgrade_with_proration(user_id: str, new_tier: str):
    """Upgrade subscription with prorated credit."""
    current_tier = SubscriptionManager(db).get_user_tier(user_id)

    if current_tier == SubscriptionTier.PRO:
        old_price = 9.99
    elif current_tier == SubscriptionTier.ENTERPRISE:
        old_price = 99.99
    else:
        old_price = 0

    if new_tier == "pro":
        new_price = 9.99
    elif new_tier == "enterprise":
        new_price = 99.99
    else:
        new_price = 0

    # Calculate days remaining in billing cycle
    query = "SELECT billing_end FROM user_subscriptions WHERE user_id = %s AND status = 'active'"
    result = db.execute(query, (user_id,))
    billing_end = result.scalar()
    days_remaining = (billing_end - datetime.now()).days

    # Calculate prorated amount
    daily_old = old_price / 30
    daily_new = new_price / 30
    credits = (daily_new - daily_old) * days_remaining

    return {
        "old_tier": current_tier.value,
        "new_tier": new_tier,
        "credit": credits,
        "next_billing": billing_end
    }
```


## Monetization Best Practices

### Pricing

- [ ] Start with simple 2-3 tier model
- [ ] Price between competitors
- [ ] Offer annual discount (20-30%)
- [ ] A/B test pricing regularly
- [ ] Clear tier comparison
- [ ] Trial period (14 days) for conversions

### Feature Gating

- [ ] Gate at API level (not just UI)
- [ ] Clear error messages on limit
- [ ] Show upgrade path in error
- [ ] Track feature usage per tier
- [ ] Monitor feature adoption
- [ ] Adjust limits based on usage data

### Payments

- [ ] Accept major payment methods
- [ ] Clear billing communication
- [ ] Easy upgrade/downgrade
- [ ] Automate renewal reminders
- [ ] Handle failed payments gracefully
- [ ] Transparent pricing (no hidden fees)

### Analytics

- [ ] Track MRR weekly
- [ ] Monitor churn rate
- [ ] Calculate LTV
- [ ] Analyze upgrade funnel
- [ ] Cohort analysis by signup date
- [ ] Tier-specific engagement metrics

### Growth

- [ ] Free tier acquisition
- [ ] Pro tier power users (10-30% conversion)
- [ ] Enterprise team focus
- [ ] Reduce churn with engagement
- [ ] Win-back campaigns for canceled
- [ ] Referral incentives
